In [ ]:
import scipy.stats as st
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

### Manual annotation

In [ ]:
output_dir = "" # Fill this with your local output dir
session_dir = f"{output_dir}/session_c"
if not os.path.isdir(session_dir):
    os.makedirs(session_dir)

In [ ]:
mapping_list= pd.read_csv("../session_c/pbmc_facs_genes.csv")
mapping_list

In [ ]:
gene_to_symbols = dict(
    zip(mapping_list['ensembl_id'], mapping_list['gene_symbol']))
gene_to_symbols

In [ ]:
adata = sc.read_h5ad(f"{output_dir}/session_b/adata_vst_final.h5ad")
adata

In [ ]:
adata.var_names = (adata.var_names.map(gene_to_symbols))

In [ ]:
sc.pl.umap(
    adata,
    color="leiden",
    title="Seurat vst best Leiden",
    legend_loc="on data"
)

In [ ]:
sc.pl.umap(
    adata,
    color="CD4",
    cmap="viridis",
    title="CD4 expression"
)

### Manual annotation by inspecting degs

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
degs_all=sc.get.rank_genes_groups_df(adata,group=None) 
degs=sc.get.rank_genes_groups_df(adata,group=None,pval_cutoff=0.01) 
degs

In [ ]:
top2 = (degs.groupby("group").head(2))
top2

In [ ]:
genes_to_plot = top2["names"].unique().tolist()

In [ ]:
sc.pl.umap(
    adata,
    color=genes_to_plot,
    cmap="viridis",
    ncols=2
)

In [ ]:
sc.pl.stacked_violin(adata,genes_to_plot, groupby="leiden", swap_axes=True)

In [ ]:
cluster = "3"
volcano_df = degs_all[degs_all.group == cluster]
volcano_df

In [ ]:
logfc_cutoff = 1
pval_cutoff = 0.01
volcano_df["category"] = "Not significant"

In [ ]:
volcano_df.loc[
    (volcano_df["logfoldchanges"] > logfc_cutoff) &
    (volcano_df["pvals_adj"] < pval_cutoff),
    "category"
] = "Upregulated"

volcano_df.loc[
    (volcano_df["logfoldchanges"] < -logfc_cutoff) &
    (volcano_df["pvals_adj"] < pval_cutoff),
    "category"
] = "Downregulated"

In [ ]:
volcano_df

In [ ]:
colors = {
    "Not significant": "lightgray",
    "Upregulated": "green",
    "Downregulated": "blue"
}

plt.figure(figsize=(8,6))
for category, color in colors.items():
    subset = volcano_df[
        volcano_df["category"] == category
    ]

    plt.scatter(
        subset["logfoldchanges"],
        -np.log10(subset["pvals_adj"]),
        c=color,
        s=10,
        alpha=0.7,
        label=category
    )

plt.axvline(logfc_cutoff, linestyle="--", color="black")
plt.axvline(-logfc_cutoff, linestyle="--", color="black")
plt.axhline(-np.log10(pval_cutoff), linestyle="--", color="orange")
plt.xlabel("Log fold change")
plt.ylabel("-log10 adjusted p-value")
plt.title("Volcano plot")
plt.legend()
plt.show()

### Based on Enrichment how could we do the annotation? 

In [ ]:
def readAnnotations():
    data = {}
    class_names = {}
    universe = set()

    with open("../session_c/sctype_annotations.csv") as f:
        for line in f:
            line = line.strip().split("|")
            gene = line[0]
            class_id = line[1]
            name = line[2]

            universe.add(gene)

            if class_id not in data:
                data[class_id] = set()

            if class_id not in class_names:
                class_names[class_id] = name

            data[class_id].add(gene)

    return data, class_names, universe

In [ ]:
degs["names"].dropna().unique().tofile(
    "../session_c/degs_genes.txt",
    sep="\n"
)

In [ ]:
def readGenes():
    genelist=set()
    f=open("../session_c/degs_genes.txt",'r')
    for line in f:
        gene=line.strip()
        genelist.add(gene)
        universe.add(gene)
    f.close()

    return genelist

In [ ]:
classes, class_names, universe = readAnnotations()
genelist = readGenes()
universe=universe | genelist

In [ ]:
deg_genes_by_cluster = {
    group: set(degs.loc[degs["group"] == group, "names"].dropna())
    for group in degs["group"].unique()
}
deg_genes_by_cluster

In [ ]:
classes

In [ ]:
results = []
for cluster, genelist in deg_genes_by_cluster.items():
    for cat in classes:
        Cl = classes[cat]

        a = len(genelist & Cl)
        b = len(genelist - Cl)
        c = len(Cl - genelist)
        d=len(universe-genelist-Cl)

        matrix = [[a, b], [c, d]]
        pvalue=st.fisher_exact(matrix,'greater')

        results.append({
            "cluster": cluster,
            "cell_type": cat,
            "pvalue": pvalue[1]
        })

enrichment_df = pd.DataFrame(results)
enrichment_df

In [ ]:
best_annotations = (
    enrichment_df
    .sort_values(["cluster", "pvalue"])
    .groupby("cluster")
    .head(1)
)

best_annotations

In [ ]:
cluster_to_celltype = dict(
    zip(
        best_annotations["cluster"],
        best_annotations["cell_type"]
    )
)

cluster_to_celltype

In [ ]:
adata.obs["cell_type_annotation"] = (
    adata.obs["leiden"]
    .map(cluster_to_celltype)
)

In [ ]:
adata.obs["cell_type_annotation"] 

In [ ]:
sc.pl.umap(
    adata,
    color="cell_type_annotation",
    legend_loc="on data",
    legend_fontsize=7
)